# Phase 5D — Unsupervised Learning

**Algorithms:** K-Means clustering, DBSCAN, PCA (dimensionality reduction), t-SNE (visualization).

**Key difference from supervised learning:** No labels — the model finds hidden structure in data.

**Install:** `pip install scikit-learn`

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs, make_moons, load_iris

np.random.seed(42)
sns.set_theme(style="whitegrid")

---
## 1. K-Means Clustering

**How it works:** Choose k centroids → assign each point to nearest centroid → move centroid to cluster mean → repeat until convergence.

**You must choose k in advance.** Use the Elbow method or Silhouette score.

In [ ]:
# Generate synthetic clusters
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Elbow method — find optimal k
inertias = []
silhouettes = []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_range, inertias, "o-", color="steelblue")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia (within-cluster sum of squares)")
axes[0].set_title('Elbow Method — look for the "elbow" point')

axes[1].plot(k_range, silhouettes, "o-", color="coral")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score — higher = better separated")

plt.tight_layout()
plt.show()
print(f"Best k by silhouette: {k_range[np.argmax(silhouettes)]}")

In [ ]:
# Fit K-Means with k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap="Set2", s=30, alpha=0.7)
axes[0].set_title("True Labels")

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap="Set2", s=30, alpha=0.7)
axes[1].scatter(
    km.cluster_centers_[:, 0],
    km.cluster_centers_[:, 1],
    c="red",
    marker="X",
    s=200,
    zorder=5,
    label="Centroids",
)
axes[1].set_title(f"K-Means (k=4) — Silhouette={silhouette_score(X_blobs, labels):.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. DBSCAN — Density-Based Clustering

Unlike K-Means:
- Finds **arbitrarily shaped** clusters
- Automatically identifies **outliers** (labeled -1)
- **No need to specify k**
- Works poorly when cluster densities vary widely

In [ ]:
# Moon-shaped clusters — K-Means fails here, DBSCAN excels
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

km_moon = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_moon.fit_predict(X_moons)

db = DBSCAN(eps=0.2, min_samples=5)
labels_db = db.fit_predict(X_moons)
n_noise = np.sum(labels_db == -1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km, cmap="Set1", s=30)
axes[0].set_title("K-Means (fails on non-convex shapes)")

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db, cmap="Set1", s=30)
axes[1].set_title(f"DBSCAN (eps=0.2, min_samples=5) — {n_noise} outliers detected")

plt.tight_layout()
plt.show()
print(f"DBSCAN unique labels: {np.unique(labels_db)}  (-1 = outlier)")

---
## 3. PCA — Principal Component Analysis

**Purpose:** Reduce many features to fewer dimensions while preserving the most variance.

**Uses in ML:**
- Speed up training by reducing features
- Visualize high-dimensional data in 2D/3D
- Remove multicollinearity

In [ ]:
iris = load_iris()
X_iris = iris.data  # 4 features: sepal length/width, petal length/width
y_iris = iris.target
target_names = iris.target_names

# Always scale before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# PCA — fit to see how many components to keep
pca_full = PCA()
pca_full.fit(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 5), explained * 100, color="steelblue", label="Individual")
axes[0].plot(range(1, 5), cumulative * 100, "ro-", label="Cumulative")
axes[0].axhline(95, color="green", linestyle="--", label="95% threshold")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance (%)")
axes[0].set_title("PCA — Explained Variance")
axes[0].legend()

# Project to 2D for visualization
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

for label, name, color in zip([0, 1, 2], target_names, ["steelblue", "coral", "green"]):
    mask = y_iris == label
    axes[1].scatter(
        X_pca[mask, 0], X_pca[mask, 1], label=name, s=40, alpha=0.8, color=color
    )

axes[1].set_xlabel(f"PC1 ({explained[0]:.1%} variance)")
axes[1].set_ylabel(f"PC2 ({explained[1]:.1%} variance)")
axes[1].set_title("Iris in 2D PCA Space")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\n2 components explain {cumulative[1]:.1%} of variance")

---
## 4. t-SNE — Visualizing High-Dimensional Data

t-SNE is for **visualization only** (not feature engineering). It finds a 2D/3D embedding that preserves local neighborhood structure.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for label, name, color in zip([0, 1, 2], target_names, ["steelblue", "coral", "green"]):
    mask = y_iris == label
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1], label=name, s=40, alpha=0.8, color=color
    )
    axes[1].scatter(
        X_tsne[mask, 0], X_tsne[mask, 1], label=name, s=40, alpha=0.8, color=color
    )

axes[0].set_title("PCA (preserves global structure)")
axes[1].set_title("t-SNE (preserves local structure)")
for ax in axes:
    ax.legend()

plt.suptitle("Iris Dataset: PCA vs t-SNE", fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

| Method | Type | Use Case |
|--------|------|----------|
| K-Means | Clustering | Well-separated, convex clusters; customer segmentation |
| DBSCAN | Clustering | Irregular shapes, outlier detection |
| PCA | Dim reduction | Speed up training, remove collinearity |
| t-SNE | Visualization | Understand high-dim data structure (NOT for features) |